In [1]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("/content/drive/MyDrive/PM25_project")
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUTPUT_DIR / "model_ready_dataset.csv"

model_df = pd.read_csv(
    model_path,
    parse_dates=["Date", "Target_Date"]
)

print("Dataset loaded:", model_path.exists())
print("Shape:", model_df.shape)

print("\nRows per split:")
print(model_df["Split"].value_counts())

display(model_df.head())


Mounted at /content/drive
Dataset loaded: True
Shape: (11037, 22)

Rows per split:
Split
train         7751
validation    2188
test          1098
Name: count, dtype: int64


,City,Date,Target_Date,Split,PM2.5,PM2.5_lag_1,PM2.5_lag_2,PM2.5_lag_3,PM2.5_lag_7,PM2.5_roll_mean_3,...,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,PM2.5_next_day,baseline_comparable
0,Bengaluru,2015-03-20,2015-03-21,train,30.65,NaN,NaN,NaN,NaN,NaN,...,28.48,18.68,27.20,4.55,2.76,43.25,36.00,171.22,48.59,False
1,Bengaluru,2015-03-21,2015-03-22,train,48.59,30.65,NaN,NaN,NaN,NaN,...,27.00,18.04,28.02,3.89,1.94,52.96,21.33,196.72,47.38,False
2,Bengaluru,2015-03-22,2015-03-23,train,47.38,48.59,30.65,NaN,NaN,42.206667,...,22.39,15.33,23.19,11.29,2.05,71.85,14.97,138.20,65.65,False
3,Bengaluru,2015-03-23,2015-03-24,train,65.65,47.38,48.59,30.65,NaN,53.873333,...,26.35,17.45,27.76,9.95,6.30,72.84,9.88,100.77,60.47,False
4,Bengaluru,2015-03-24,2015-03-25,train,60.47,65.65,47.38,48.59,NaN,57.833333,...,29.87,20.88,35.10,1.46,6.07,64.12,5.90,61.48,62.56,False


### Day 6 — Task 1: Training-Only Missingness Check

I will calculate the missing-value percentage for each candidate predictor using only the training split (target dates from 2015–2018). Any predictor with more than 40% missing values in the training set will be excluded.

This check must use training data only to avoid using information from the validation or test periods when selecting features. Missing values in the remaining numeric predictors will be handled later using a preprocessing pipeline fitted on the training set.

In [3]:

# Metadata and target columns are not predictor features
excluded_cols = [
    "City", "Date", "Target_Date", "Split",
    "PM2.5_next_day", "baseline_comparable"
]

candidate_features = [
    col for col in model_df.columns
    if col not in excluded_cols
]

# Use training data only
train_df = model_df.loc[model_df["Split"] == "train"].copy()

missing_pct = (
    train_df[candidate_features]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missingness_table = missing_pct.rename(
    "Missing_Percentage"
).to_frame()

display(missingness_table.round(2))

# Drop features with MORE than 40% missingness in training
dropped_features = missing_pct[missing_pct > 40].index.tolist()

selected_features = [
    col for col in candidate_features
    if col not in dropped_features
]

print("Features dropped (>40% missing in training):", dropped_features)
print("Number of candidate features:", len(candidate_features))
print("Number of selected features:", len(selected_features))
print("Selected features:", selected_features)

assert all(missing_pct[f] <= 40 for f in selected_features)
assert all(missing_pct[f] > 40 for f in dropped_features)

print("\nTraining-only feature selection checks passed.")


,Missing_Percentage
NH3,34.50
Benzene,7.66
Toluene,6.08
NOx,3.99
PM2.5_roll_mean_7,3.91
O3,2.83
PM2.5_lag_7,2.12
SO2,1.47
PM2.5_roll_mean_3,1.35
PM2.5_lag_3,1.35


Features dropped (>40% missing in training): []
Number of candidate features: 16
Number of selected features: 16
Selected features: ['PM2.5', 'PM2.5_lag_1', 'PM2.5_lag_2', 'PM2.5_lag_3', 'PM2.5_lag_7', 'PM2.5_roll_mean_3', 'PM2.5_roll_mean_7', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene']

Training-only feature selection checks passed.


### Day 6 — Task 2: Leak-Free Preprocessing Pipeline

I will build a scikit-learn preprocessing pipeline for the selected numeric predictors and the categorical `City` feature. Numeric missing values will be filled using the median calculated from the training set, followed by standard scaling. City will be one-hot encoded, with unseen categories handled using `handle_unknown="ignore"`.

The preprocessing transformer will be fitted only on the training split. Validation and test data will not be used to calculate imputation medians or scaling parameters.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numeric and categorical predictors
numeric_features = selected_features
categorical_features = ["City"]

# Numeric preprocessing: median imputation, then scaling
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing: one-hot encoding
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine both preprocessing paths
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("city", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

# Separate the data by split
train_df = model_df.loc[model_df["Split"] == "train"].copy()
val_df = model_df.loc[model_df["Split"] == "validation"].copy()

X_train = train_df[numeric_features + categorical_features]
y_train = train_df["PM2.5_next_day"]

X_val = val_df[numeric_features + categorical_features]
y_val = val_df["PM2.5_next_day"]

# Fit preprocessing on training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Transform validation data without fitting on it
X_val_processed = preprocessor.transform(X_val)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Preprocessing fitted on training data only.")

Training rows: 7751
Validation rows: 2188
Processed training shape: (7751, 22)
Processed validation shape: (2188, 22)
Preprocessing fitted on training data only.


### Day 6 — Task 3: Verify Training-Only Imputation

I will compare the medians stored by the fitted numeric imputer against the medians calculated directly from the training split. I will also compare them with medians from the full dataset to demonstrate why fitting preprocessing only on training data is important.

The full-dataset medians are used only for this diagnostic comparison; they will not be used to fit or modify the preprocessing pipeline.

In [5]:
# Retrieve medians learned by the fitted imputer
imputer = preprocessor.named_transformers_["num"].named_steps["imputer"]
fitted_medians = pd.Series(
    imputer.statistics_,
    index=numeric_features,
    name="Fitted_Imputer_Median"
)

# Calculate medians independently
training_medians = (
    train_df[numeric_features]
    .median()
    .rename("Training_Median")
)

full_data_medians = (
    model_df[numeric_features]
    .median()
    .rename("Full_Dataset_Median")
)

# Compare all three
median_check = pd.concat(
    [fitted_medians, training_medians, full_data_medians],
    axis=1
)

median_check["Matches_Training"] = np.isclose(
    median_check["Fitted_Imputer_Median"],
    median_check["Training_Median"],
    equal_nan=True
)

display(median_check.round(4))

# Confirm the fitted imputer learned training medians
assert median_check["Matches_Training"].all()

print("PASS: All fitted imputer medians match the training medians.")
print(
    "Features where training and full-dataset medians differ:",
    int(
        (
            ~np.isclose(
                median_check["Training_Median"],
                median_check["Full_Dataset_Median"],
                equal_nan=True
            )
        ).sum()
    )
)

,Fitted_Imputer_Median,Training_Median,Full_Dataset_Median,Matches_Training
PM2.5,59.3400,59.3400,54.9400,True
PM2.5_lag_1,59.3800,59.3800,54.9800,True
PM2.5_lag_2,59.3000,59.3000,54.9550,True
PM2.5_lag_3,59.2800,59.2800,54.9800,True
PM2.5_lag_7,59.3200,59.3200,55.2250,True
PM2.5_roll_mean_3,59.7483,59.7483,55.8000,True
PM2.5_roll_mean_7,60.0307,60.0307,55.9743,True
NO,10.4000,10.4000,9.9800,True
NO2,28.0700,28.0700,27.4250,True
NOx,20.1350,20.1350,21.8600,True


PASS: All fitted imputer medians match the training medians.
Features where training and full-dataset medians differ: 16


In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Validation rows where the 7-day mean is available
baseline_val = val_df.loc[
    val_df["baseline_comparable"].astype(bool)
].copy()

# Actual next-day PM2.5
y_true = baseline_val["PM2.5_next_day"]

# Baseline predictions
y_pred_persistence = baseline_val["PM2.5"]
y_pred_7day = baseline_val["PM2.5_roll_mean_7"]

# Calculate metrics
baseline_results = pd.DataFrame([
    {
        "Model": "Persistence",
        "Rows_Evaluated": len(baseline_val),
        "MAE": mean_absolute_error(y_true, y_pred_persistence),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred_persistence)
        )
    },
    {
        "Model": "7-day Mean",
        "Rows_Evaluated": len(baseline_val),
        "MAE": mean_absolute_error(y_true, y_pred_7day),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred_7day)
        )
    }
])

display(baseline_results.round(3))

print("Validation period: 2019")
print("Common evaluation rows:", len(baseline_val))

,Model,Rows_Evaluated,MAE,RMSE
0,Persistence,2182,16.376,29.857
1,7-day Mean,2182,21.233,37.134


Validation period: 2019
Common evaluation rows: 2182


In [7]:
baseline_path = OUTPUT_DIR / "baseline_results_validation.csv"

baseline_results.to_csv(baseline_path, index=False)

print("Saved baseline results to:", baseline_path)
print("File exists:", baseline_path.exists())

Saved baseline results to: /content/drive/MyDrive/PM25_project/outputs/baseline_results_validation.csv
File exists: True


In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Use validation rows only
classification_val = val_df.copy()

# Actual next-day class: 1 = Poor, 0 = Not Poor
y_true_class = (
    classification_val["PM2.5_next_day"] > 90
).astype(int)

# Baseline prediction from current-day PM2.5
y_pred_class = (
    classification_val["PM2.5"] > 90
).astype(int)

# Calculate classification metrics
classification_results = pd.DataFrame([{
    "Model": "Current-day threshold baseline",
    "Threshold": 90,
    "Rows_Evaluated": len(classification_val),
    "Precision": precision_score(
        y_true_class, y_pred_class, zero_division=0
    ),
    "Recall": recall_score(
        y_true_class, y_pred_class, zero_division=0
    ),
    "F1": f1_score(
        y_true_class, y_pred_class, zero_division=0
    )
}])

display(classification_results.round(4))

,Model,Threshold,Rows_Evaluated,Precision,Recall,F1
0,Current-day threshold baseline,90,2188,0.8398,0.8398,0.8398


In [9]:
# Save classification baseline results
classification_path = OUTPUT_DIR / "classification_baseline_validation.csv"

classification_results.to_csv(classification_path, index=False)

print("Saved to:", classification_path)
print("File exists:", classification_path.exists())

# Verify the saved file
display(pd.read_csv(classification_path))

Saved to: /content/drive/MyDrive/PM25_project/outputs/classification_baseline_validation.csv
File exists: True


,Model,Threshold,Rows_Evaluated,Precision,Recall,F1
0,Current-day threshold baseline,90,2188,0.839757,0.839757,0.839757
